## Developing notes - overview

In [12]:
# CO SE TU DĚJE

# Zadáme:
## - procentuální scénář pro tvorbu kartogramů (např 50%)
## - požadovanou statistickou metodu (medián nebo průměr)
## - rok ke kterému přepočteme inflaci

# Načteme "Detailed cases" data (případy sanací, kde známe cenu)
## U těchto případů přepočítáme ceny na ceny aktuální (zohledníme inflaci)
## Započítáváme všechny případy dle typů původce znečištění
### - ty, ke kterým máme použitelá data (aspoň 3 případy a aspoň 2% celkových dat: je to 5 kategorií) započítáváme detailně zvlášť
### - všechny ostatní zahrneme do kategorie "ostatní"
### Spočítáme průměr a medián cen (co případ to stejná váha)

# Načteme export ze SEKM
## Počítáme se všemi kategoriemi dle typů původce znečištění (5 detailních + ostatní)
## Přidáme k nim průměr a medián cen dle "Detailed cases"
## Zagregujeme 1) dle krajů 2) dle krajů a typů původce znečištění

# Výsledkem jsou kartogramy

# -----------------------------
# CO SE JEŠTĚ MÁ DÍT

## DROBNOSTI:
### někam do mapového pole celkový součet částky
### bokem mapového pole - vypínatelně - tabulku částek ke krajům

## SEKM filtr dle "P", "N"
## souhrnná funkce tvořící kartogramy (a správně je pojmenovává) a ukládá je do složky XY (vytvoří složku a pojmenuje ji i s dnešním datem)
## zhezčit kod - never ending story

In [13]:
# NICE TO HAVE -
## Vytvořit kartogramy po ORP
## ORP cartogram fce - opravit - správněji pojmenovat

In [14]:
# USING DATA:
# Detail cases data: Data_opzp_6_08_04.xlsx

## Global variables

In [15]:
SELECTED_STAT_METHOD = "medián" # medián nebo průměr

MY_PERCENTAGE_SCENARIO = 50 # Zvol scénář: modifikace vstupních dat do kartogramu. Zadáváme v %.

YEAR_TO_COUNT_INFLATION = 2025

# KATEGORIE PRIORITY: - všechny v současném SEKM souboru ['A1.0', 'A1.1', 'A1.2', 'A1.3', 'A2.0', 'A2.1', 'A2.2', 'A2.3', 'A3.1', 'A3.2', 'A3.3', 'N0.0', 'N0.1', 'N0.2', 'N1.0',
# 'N1.1', 'N1.2', 'N2.0', 'N2.1', 'N2.2', 'N2.3', 'P1.0', 'P1.1', 'P1.2', 'P1.3', 'P2.0', 'P2.1', 'P2.2', 'P2.3', 'P3.0', 'P3.1', 'P3.2', 'P3.3', 'P4.0', 'P4.1', 'P4.2', 'P4.3']

SAVE_EXCEL_INFLATION = 0 # wanna save simplified excel with inflation data?  # --- to desktop (1 = save, everything else = dont save)

SAVE_CARTOGRAMS = False # Do I save cartograms?

## Imports, functions, dictionaries

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors
import openpyxl

# for cartograms
import geopandas as gpd
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

### Dictionaries

In [17]:
cr_inflation = {2013:1.4,
                2014:0.4,
                2015:0.3,
                2016:0.7,
                2017:2.5,
                2018:2.1,
                2019:2.8,
                2020:3.2,
                2021:3.8,
                2022:15.1,
                2023:10.7,
                2024:2.4,
                2025:2.5}

# data from ČSÚ - February 2026

### Functions

In [18]:
# print locality type info
def print_locality_type_info(selected_code, my_dict, my_counts):
    for key, value in my_dict.items():
        if value == selected_code:
            count = my_counts.get(key, 0)
            print(f"locality type number: {value} / name: {key} / row count: {count}")
            return

    print("Code not found.")

In [19]:
# print df info:  cols, rows
def print_df_info(my_df):
    num_cols = my_df.shape[1]
    num_rows = my_df.shape[0]
    print("Number of columns:", num_cols)
    print("Number of rows:", num_rows)

### Inflation counter

In [20]:
# cpi = consumer price index (index spotřebitelských cen)
cpi = pd.Series(cr_inflation).sort_index()
cpi = (1 + cpi/100).cumprod()
my_min = min(cr_inflation) - 1 # základní hodnota o rok nižší než nejnižší rok.
cpi.loc[my_min] = 1
cpi = cpi.sort_index()
cpi_dict = cpi.to_dict()

# inflation counting function
def adjust_by_inflation(starting_value, year_from, year_to):

    if year_from not in cpi_dict  or year_to not in cpi_dict: # if wrong year
            raise ValueError("Year not available in CPI index")

    output_value = starting_value * cpi_dict[year_to] / cpi_dict[year_from] # count
    return output_value

#### Inflation counter - testing

In [21]:
adjust_by_inflation(100,2021,2025)

133.73551872000002

## SEKM data A)
#### load data, basic DF info

In [22]:
# load data
My_SEKM_data = r"C:\Users\matej.piro\Desktop\PROJEKTY\Suchánek_quest\info_05_02\Export_celý_SEKM_bez_vyloučených_05022026_Matej.xlsx"
mySEKM_DF = pd.read_excel(My_SEKM_data)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\matej.piro\\Desktop\\PROJEKTY\\Suchánek_quest\\info_05_02\\Export_celý_SEKM_bez_vyloučených_05022026_Matej.xlsx'

In [ ]:
# basic DF info
print_df_info(mySEKM_DF)
mySEKM_DF.head(1)

## Detailed cases - data

### Load data

In [ ]:
# LOAD DATA
My_data = r"C:\Users\matej.piro\Desktop\PROJEKTY\Suchánek_quest\Data_update_04_08\Data_opzp_6_08_04.xlsx"
my_DF = pd.read_excel(My_data)

my_DF.head(1)

### Basic data overview

In [ ]:
print_df_info(my_DF)

In [ ]:
print(my_DF["odhad_nakladu_cela_lokalita_(rok)"].min())
print(my_DF["odhad_nakladu_cela_lokalita_(rok)"].max())

### Inflation
#####  - count inflation, add new column to DF

In [ ]:
# max year in dataframe
year_to = YEAR_TO_COUNT_INFLATION

# name of new column
new_col = f"odhad_nakladu_cela_lokalita_(Kc)_inf_{year_to}"

# map CPI index to each row
cpi_from = my_DF["odhad_nakladu_cela_lokalita_(rok)"].map(cpi_dict) # .map() is way faster than .apply() which is another possible solution

# compute adjusted values
inflated_values = (
    my_DF["odhad_nakladu_cela_lokalita_(Kc)"] *
    cpi_dict[year_to] /
    cpi_from
)

# position where to insert the new column
new_col_position = my_DF.columns.get_loc("odhad_nakladu_cela_lokalita_(Kc)") + 1

# insert column
my_DF.insert(new_col_position, new_col, inflated_values)

In [ ]:
my_DF.head(2)

#### KC column format

In [ ]:
# format kc columns?
FORMAT_KC_COLUMNS = 0

# dynamicaly created column
my_inflation_col = f"odhad_nakladu_cela_lokalita_(Kc)_inf_{YEAR_TO_COUNT_INFLATION}"

# select columns to output
my_DF_selected_1 = my_DF[["lokalita", "typ_puvodce_znecisteni", "odhad_nakladu_cela_lokalita_(Kc)", "odhad_nakladu_cela_lokalita_(rok)",my_inflation_col]]

# compute diff in CZK and diff in %
my_DF_selected_1["cena_diff_(Kc)"] = my_DF_selected_1[my_inflation_col] - my_DF_selected_1["odhad_nakladu_cela_lokalita_(Kc)"]
my_DF_selected_1["cena_diff_(%)"] = (((my_DF_selected_1[my_inflation_col] / my_DF_selected_1["odhad_nakladu_cela_lokalita_(Kc)"]) * 100) - 100).round(1)

# format certain columns nicely - string!
if FORMAT_KC_COLUMNS == 1:

    cols_to_format = [
        "odhad_nakladu_cela_lokalita_(Kc)",
        my_inflation_col,
        "cena_diff_(Kc)"]

    for col in cols_to_format:
        my_DF_selected_1[col] = my_DF_selected_1[col].map("{:,.2f}".format)

# display
my_DF_selected_1.head(2)

#### Save excel

In [ ]:
if SAVE_EXCEL_INFLATION == 1:

    my_DF_selected_1.to_excel(
        "C:/Users/matej.piro/Desktop/my_DF_selected_1.xlsx",
        index=False
    )
    print("Excel created")

else:
    print("No excel for you now")

### Compute MEDIAN, AVG per selected categories
##### žádná váha dle rozlohy

In [23]:
# přidá kategorii "ostatní" kam shrne vše mimo selected_types_puvodce_znecisteni_types
def count_mean_median_for_sel_types(df, selected_types):
    '''group by typ_puvodce_znecisteni, count mean, median,
    add nick column, returns df'''

    # původní vybrané kategorie
    result_df = df[
        df["typ_puvodce_znecisteni"].isin(selected_types)
    ].groupby(
        "typ_puvodce_znecisteni"
    )[
        "odhad_nakladu_cela_lokalita_(Kc)_inf_2025"
    ].agg(["count", "mean", "median"])

    # souhrnná kategorie "ostatní"
    ostatni_df = df[
        ~df["typ_puvodce_znecisteni"].isin(selected_types)
    ][
        "odhad_nakladu_cela_lokalita_(Kc)_inf_2025"
    ].agg(["count", "mean", "median"])

    result_df.loc["ostatní"] = ostatni_df

    # add nick column
    result_df["nick"] = result_df.index.str[:4]
    result_df = result_df[["nick", "count", "mean", "median"]]

    return result_df

In [24]:
# list of selected categories
selected_types_puvodce_znecisteni_types = ["strojírenství",
                                         "plynárenství",
                                         "chemický průmysl (léčiva, gumárenství, plasty, umělá vlákna...)",
                                         "doprava a distribuce (produktovody, distribuční sklady)",
                                         "hutnictví a slévárenství",
                                         "ostatní"
                                        ]

In [25]:
# call function, visualise
detailed_cases_result_df = count_mean_median_for_sel_types(my_DF_selected_1,selected_types_puvodce_znecisteni_types)

detailed_cases_result_df.head(6)

NameError: name 'my_DF_selected_1' is not defined

## SEKM data B)

#### SEKM DF info

In [ ]:
print_df_info(mySEKM_DF)
mySEKM_DF.head(1)

##### SEKM DF typ_puvodce_znecisteni info

In [ ]:
# COUNT BY: typ_puvodce_znecisteni

# map a number to each typ_puvodce_znecisteni type
categories = sorted(
    mySEKM_DF["typ_puvodce_znecisteni"].dropna().unique()
)

# dictionary
mapping_dict_SEKM = {cat: i + 1 for i, cat in enumerate(categories)}

# new column "typ_puvodce_znecisteni_code" into DF on second index
mySEKM_DF.insert(
    1,
    "typ_typ_puvodce_znecisteni_code",
    mySEKM_DF["typ_puvodce_znecisteni"].map(mapping_dict_SEKM)
)

# get counts of each typ_puvodce_znecisteni type
counts = mySEKM_DF["typ_puvodce_znecisteni"].value_counts()

# print dictionary with counts
for key, value in mapping_dict_SEKM.items():
    count = counts.get(key, 0)
    print(f"{value} - {key} (count: {count})")

### test - Try filtering

In [ ]:
# FILTER:
## typ_puvodce_znecisteni = strojírenství
## kategorie priority začíná na P
## Kraj = Jihomoravský
# my_selected_SEKM_df = mySEKM_DF[
#     (mySEKM_DF["typ_puvodce_znecisteni"] == "strojírenství") &
#     (mySEKM_DF["Kategorie priority"].str.startswith("P", na=False)) &
#     (mySEKM_DF["Kraj"] == "Jihomoravský kraj")
# ]

my_selected_SEKM_df = mySEKM_DF[
    (mySEKM_DF["typ_puvodce_znecisteni"].isin(selected_types_puvodce_znecisteni_types)) &
    (mySEKM_DF["Kraj"] == "Ústecký kraj")
]

print_df_info(my_selected_SEKM_df)

In [ ]:
my_selected_SEKM_df.head(1)

In [ ]:
my_selected_SEKM_df_short = my_selected_SEKM_df[["Název lokality", "Kraj", "ORP", "typ_puvodce_znecisteni", "Plocha m2", "Kategorie priority"]]
my_selected_SEKM_df_short.head(1)

### Modifying mySEKM_DF into mySEKM_modified_DF
##### "typ_puvodce_znecisteni" vše krom pěti vybraných (z "selected_types_puvodce_znecisteni_types") do "ostatní"

In [ ]:
mySEKM_modified_DF = mySEKM_DF.copy()

mySEKM_modified_DF.loc[
    ~mySEKM_modified_DF["typ_puvodce_znecisteni"].isin(selected_types_puvodce_znecisteni_types),
    "typ_puvodce_znecisteni"
] = "ostatní"

In [ ]:
# zobraz
mySEKM_modified_DF["typ_puvodce_znecisteni"].value_counts()

### Add columns with computed values to SEKM df

In [ ]:
# 24.6. SMAŽ
detailed_cases_result_df.head(8)

In [ ]:
# adding columns
mySEKM_modified_DF["pred_mean_price"] = mySEKM_modified_DF["typ_puvodce_znecisteni"].map(detailed_cases_result_df["mean"])
mySEKM_modified_DF["pred_median_price"] = mySEKM_modified_DF["typ_puvodce_znecisteni"].map(detailed_cases_result_df["median"])

In [26]:
# how many rows were afected
mask = mySEKM_modified_DF["typ_puvodce_znecisteni"].isin(selected_types_puvodce_znecisteni_types)

count_match = mask.sum()
count_other = (~mask).sum()

print("Našich pět typů původce znečištění reprezentuje:", count_match, "řádků.")
print("Nepoužíváme:", count_other, "řádků.")

NameError: name 'mySEKM_modified_DF' is not defined

### Count and output per "Kraje"

In [ ]:
# filter by selected_types_puvodce_znecisteni_types
selected_typ_puvodce_znecisteni_df = mySEKM_modified_DF[
    mySEKM_modified_DF["typ_puvodce_znecisteni"].isin(selected_types_puvodce_znecisteni_types)
]

##### by Kraj and typ_puvodce_znecisteni

In [ ]:
# group by Kraj and typ_puvodce_znecisteni
Kraje_a_typy_puvodce_price_pred_df = selected_typ_puvodce_znecisteni_df.groupby(
    ["Kraj", "typ_puvodce_znecisteni"]
)[["pred_mean_price", "pred_median_price"]].sum().reset_index()

In [ ]:
# prices to mld CZK
Kraje_a_typy_puvodce_price_pred_df["pred_mean_price(mld_CZK)"] = Kraje_a_typy_puvodce_price_pred_df["pred_mean_price"] / 1000000000
Kraje_a_typy_puvodce_price_pred_df["pred_median_price(mld_CZK)"] = Kraje_a_typy_puvodce_price_pred_df["pred_median_price"] / 1000000000

# drop columns
Kraje_a_typy_puvodce_price_pred_df = Kraje_a_typy_puvodce_price_pred_df.drop(["pred_mean_price","pred_median_price"], axis=1)

In [ ]:
Kraje_a_typy_puvodce_price_pred_df.head(1)

##### by ORP and typ_puvodce_znecisteni

In [ ]:
# group by ORP and typ_puvodce_znecisteni
ORP_a_typy_puvodce_price_pred_df = selected_typ_puvodce_znecisteni_df.groupby(
    ["ORP", "typ_puvodce_znecisteni"]
)[["pred_mean_price", "pred_median_price"]].sum().reset_index()

In [ ]:
# prices to mld CZK
ORP_a_typy_puvodce_price_pred_df["pred_mean_price(mld_CZK)"] = ORP_a_typy_puvodce_price_pred_df["pred_mean_price"] / 1000000000
ORP_a_typy_puvodce_price_pred_df["pred_median_price(mld_CZK)"] = ORP_a_typy_puvodce_price_pred_df["pred_median_price"] / 1000000000

# drop columns
ORP_a_typy_puvodce_price_pred_df = ORP_a_typy_puvodce_price_pred_df.drop(["pred_mean_price","pred_median_price"], axis=1)

##### ORP Filtr by only ONE Kraj

In [ ]:
# FILTRUJ JEDEN KRAJ
# ORP BY KRAJ - output in list

def ORP_list_by_kraj(df, kraj):
    orp_by_kraj_list = df.loc[
    df["Kraj"] == kraj,
    "ORP"].unique().tolist()

    if not orp_by_kraj_list:
        raise ValueError(f"Kraj '{kraj}' neexistuje.")

    return orp_by_kraj_list

In [ ]:
# ORP Ústecký kraj
ORP_ustecky = ORP_list_by_kraj(mySEKM_modified_DF,"Ústecký kraj")

In [ ]:
ORP_jeden_kraj_a_typy_puvodce_price_pred_df = ORP_a_typy_puvodce_price_pred_df[
    ORP_a_typy_puvodce_price_pred_df["ORP"].isin(ORP_ustecky)
]

In [ ]:
print_df_info(ORP_jeden_kraj_a_typy_puvodce_price_pred_df)
ORP_jeden_kraj_a_typy_puvodce_price_pred_df.head()

##### by Kraj

In [ ]:
# group by Kraj
Kraje_price_pred_df = selected_typ_puvodce_znecisteni_df.groupby(
    ["Kraj"]
)[["pred_mean_price", "pred_median_price"]].sum().reset_index()

In [ ]:
# prices to mld CZK
Kraje_price_pred_df["pred_mean_price(mld_CZK)"] = Kraje_price_pred_df["pred_mean_price"] / 1000000000
Kraje_price_pred_df["pred_median_price(mld_CZK)"] = Kraje_price_pred_df["pred_median_price"] / 1000000000

# drop columns
Kraje_price_pred_df = Kraje_price_pred_df.drop(["pred_mean_price","pred_median_price"], axis=1)

In [ ]:
Kraje_price_pred_df.head()

##### by ORP

In [ ]:
# group by Kraj
ORP_price_pred_df = selected_typ_puvodce_znecisteni_df.groupby(
    ["ORP"]
)[["pred_mean_price", "pred_median_price"]].sum().reset_index()

In [ ]:
# prices to mld CZK
ORP_price_pred_df["pred_mean_price(mld_CZK)"] = ORP_price_pred_df["pred_mean_price"] / 1000000000
ORP_price_pred_df["pred_median_price(mld_CZK)"] = ORP_price_pred_df["pred_median_price"] / 1000000000

# drop columns
ORP_price_pred_df = ORP_price_pred_df.drop(["pred_mean_price","pred_median_price"], axis=1)

##### Excel output: Kraje_a_typy_puvodce_price_pred_df, Kraje_price_pred_df

In [ ]:
# output to excel on desktop
KRAJE_EXCEL_OUTPUT = 0

if KRAJE_EXCEL_OUTPUT == 1:

    Kraje_a_typy_puvodce_price_pred_df.to_excel(
        r"C:/Users/matej.piro/Desktop/output_Kraje_a_typy_puvodce_price_pred_df.xlsx",
        index=False
    )

    Kraje_price_pred_df.to_excel(
        r"C:/Users/matej.piro/Desktop/output_Kraje_price_pred_df.xlsx",
        index=False
    )

## Cartograms

#### Prapare data

In [ ]:
# kraje id dict
kraj_map = {
    "CZ010": "Hlavní město Praha",
    "CZ020": "Středočeský kraj",
    "CZ031": "Jihočeský kraj",
    "CZ032": "Plzeňský kraj",
    "CZ041": "Karlovarský kraj",
    "CZ042": "Ústecký kraj",
    "CZ051": "Liberecký kraj",
    "CZ052": "Královéhradecký kraj",
    "CZ053": "Pardubický kraj",
    "CZ063": "Kraj Vysočina",
    "CZ064": "Jihomoravský kraj",
    "CZ071": "Olomoucký kraj",
    "CZ072": "Zlínský kraj",
    "CZ080": "Moravskoslezský kraj"
}

In [ ]:
my_geojson_kraje = r"C:\Users\matej.piro\Desktop\PROJEKTY\Radka_mapy\shape_json\kraje.json"
my_geojson_orp = r"C:\Users\matej.piro\Desktop\PROJEKTY\Radka_mapy\shape_json\orp.geojson"

my_geo_kraje_df = gpd.read_file(my_geojson_kraje)
my_geo_orp_df = gpd.read_file(my_geojson_orp)

In [ ]:
# Changes in df

# strip id column
my_geo_kraje_df["id"] = my_geo_kraje_df["id"].str[:-7]

# map kraje dict
my_geo_kraje_df["kraj"] = my_geo_kraje_df["id"].map(kraj_map)

In [ ]:
print_df_info(my_geo_kraje_df)
my_geo_kraje_df.head(1)

#### Cartogram settings

In [ ]:
my_geo_kraje_df.head(2)

##### Function

In [ ]:
# DEF FUNCTION

# stats_quantity je buď "medián" nebo "průměr"
# scenario_percent_change je parametr procentuální změny vstupních dat. Zadáváme v %

def create_kraje_cartogram(geo_kraje_df,color_list,stats_quantity,text_1_dict,text_2_dict,text_3_dict,scenario_percent_change=100,save=False):
    # checks
    if stats_quantity not in ["medián", "median", "MEDIÁN", "MEDIAN", "průměr", "prumer", "PRŮMĚR", "PRUMER"]:
        raise ValueError("stats_quantity must be 'medián' or 'průměr'")

    geo_kraje_df = geo_kraje_df.copy()

    # přepočet hodnot podle zadaného procenta
    scale_factor = scenario_percent_change / 100

    geo_kraje_df["cena_upravena"] = (geo_kraje_df["cena (mld CZK)"] * scale_factor)

    num_bins = 5
    my_colors = color_list

    # --- BINS ---
    bins = np.linspace(
        geo_kraje_df["cena_upravena"].min(),
        geo_kraje_df["cena_upravena"].max(),
        num_bins + 1
    )

    # --- COLORMAP + NORM ---
    cmap = ListedColormap(my_colors)
    norm = BoundaryNorm(bins, len(my_colors))

    # --- PLOT ---
    fig, ax = plt.subplots(figsize=(10, 10))

    praha = geo_kraje_df[geo_kraje_df["id"] == "CZ010"]
    zbytek = geo_kraje_df[geo_kraje_df["id"] != "CZ010"]

    # nejdřív vykresli velké celky
    zbytek.plot(
        column="cena_upravena",
        cmap=cmap,
        norm=norm,
        edgecolor="black",
        linewidth=0.8,
        ax=ax
    )

    # nakonec vykresli Prahu
    praha.plot(
        column="cena_upravena",
        cmap=cmap,
        norm=norm,
        edgecolor="black",
        linewidth=1.2,
        ax=ax
    )

    # --- LEGEND ---
    legend_patches = [
        Patch(
            facecolor=my_colors[i],
            edgecolor="black",
            label=f"{bins[i]:.1f}–{bins[i+1]:.1f}"
        )
        for i in range(num_bins)
    ]

    ax.legend(
        handles=legend_patches,
        title="mld CZK",
        loc="upper right",
        frameon=False
    )

    # text 1
    ax.text(**text_1_dict)

    # text 2
    ax.text(**text_2_dict)

    # text 3
    ax.text(**text_3_dict)

    plt.title(
        f"Distribuce ekologické zátěže dle databáze SEKM po krajích ČR, 2026.\n"
        f"Vypočtená dle {stats_quantity}ů cen* známých likvidací zátěže z předešlých let.\n"
        f"Scénář = {scenario_percent_change}%"
    )

    # save
    if save:  # means "if save = True"
        safe_stats = (
            stats_quantity
            .replace("á", "a")
            .replace("ů", "u")
        )

        path = (
            fr"C:/Users/matej.piro/Desktop/"
            fr"cartogram_{safe_stats}_{scenario_percent_change}pct.png"
        )

        plt.savefig(path, dpi=300, bbox_inches="tight")

    plt.show()

##### Cartogram texts, colors - dict definition

In [ ]:
# text dicts
text_1_dict = {
    "x": 17.2,
    "y": 48.5,
    "s": "Data: CENIA, SEKM \nVyhotovil: Matěj Píro, 2026",
    "fontsize": 10,
    "color": "black"
}

text_2_dict = {
    "x": 11.9,
    "y": 48.5,
    "s": "Tito \npůvodci \nznečištění:\n\n" \
    "- plynárenství,\n" \
    "- strojírenství,\n" \
    "- chemický průmysl,\n" \
    "- doprava a distribuce,\n" \
    "- hutnictví a slévárenství\n\n"
    " započteni přímo.\n"
    " Další započítáni v kategorii \"ostatní\".",

    "fontsize": 9,
    "color": "black"
}

text_3_dict = {
    "x": 14.5,
    "y": 48.5,
    "s": "*Ceny přepočteny na úroveň roku 2025.",
    "fontsize": 9,
    "color": "black"
}

# color list
my_colors_list = [
    "moccasin",
    "gold",
    "orange",
    "orangered",
    "maroon"
]

#### Create Cartograms - Kraje

In [ ]:
# map
kraj_price_map = dict(zip(
    Kraje_price_pred_df["Kraj"],
    Kraje_price_pred_df["pred_median_price(mld_CZK)"]
))

my_geo_kraje_df["cena (mld CZK)"] = my_geo_kraje_df["kraj"].map(kraj_price_map)

In [ ]:
create_kraje_cartogram(my_geo_kraje_df, my_colors_list, SELECTED_STAT_METHOD, text_1_dict, text_2_dict, text_3_dict, MY_PERCENTAGE_SCENARIO, save=SAVE_CARTOGRAMS)

In [27]:
print(
    pd.DataFrame({
        "kraj": my_geo_kraje_df["kraj"],
        "cena (mld CZK)": my_geo_kraje_df["cena (mld CZK)"] * MY_PERCENTAGE_SCENARIO / 100 # započten % scénář
    })
)

NameError: name 'my_geo_kraje_df' is not defined

## -------------------------------

## ZKOUŠKA - BODY V KRAJI

In [ ]:
my_selected_SEKM_df_short_2 = my_selected_SEKM_df[["Název lokality", "Kraj", "ORP", "typ_puvodce_znecisteni", "Plocha m2", "Kategorie priority","X","Y","Odhad nákladů [Kč, bez DPH]"]] # filtruj sloupce

my_selected_SEKM_df_short_2 = my_selected_SEKM_df_short_2[my_selected_SEKM_df_short_2["Odhad nákladů [Kč, bez DPH]"] != 0] # zahod řádky s 0

my_selected_SEKM_df_short_2.sort_values( # sort podle odhadu nákladů
    by="Odhad nákladů [Kč, bez DPH]",
    ascending=False,
    inplace=True
)

my_selected_SEKM_df_short_2.head()

In [ ]:
# koukni co tu máš za kraje
print(my_selected_SEKM_df_short_2["Kraj"].unique())

In [ ]:
# DEF FUNCTION

# stats_quantity je buď "medián" nebo "průměr"
# zobraz bodově místa k sanaci přes "my_number_of_points"

def create_kraje_cartogram_plus_points(geo_kraje_df, color_list, stats_quantity, text_1_dict, text_2_dict, text_3_dict, my_number_of_points, save=False):
    # checks
    if stats_quantity not in ["medián","průměr"]:
        raise ValueError("stats_quantity must be 'medián' or 'průměr'")


    num_bins = 5
    my_colors = color_list

    # --- BINS ---
    bins = np.linspace(
        geo_kraje_df["cena (mld CZK)"].min(),
        geo_kraje_df["cena (mld CZK)"].max(),
        num_bins + 1
    )

    # --- COLORMAP + NORM ---
    cmap = ListedColormap(my_colors)
    norm = BoundaryNorm(bins, len(my_colors))

    # --- PLOT ---
    fig, ax = plt.subplots(figsize=(10, 10))

    praha = geo_kraje_df[geo_kraje_df["id"] == "CZ010"]
    zbytek = geo_kraje_df[geo_kraje_df["id"] != "CZ010"]

    # nejdřív vykresli velké celky
    zbytek.plot(
        column="cena (mld CZK)",
        cmap=cmap,
        norm=norm,
        edgecolor="black",
        linewidth=0.8,
        ax=ax
    )

    # nakonec vykresli Prahu
    praha.plot(
        column="cena (mld CZK)",
        cmap=cmap,
        norm=norm,
        edgecolor="black",
        linewidth=1.2,
        ax=ax
    )

    body_gdf = gpd.GeoDataFrame(
    my_selected_SEKM_df_short_2,
    geometry=gpd.points_from_xy(
        my_selected_SEKM_df_short_2["X"],
        my_selected_SEKM_df_short_2["Y"]
    ),
    crs="EPSG:2065"
)

    body_gdf = body_gdf.to_crs(geo_kraje_df.crs)

    body_gdf = body_gdf.nlargest(
    my_number_of_points,
    "Odhad nákladů [Kč, bez DPH]"
)

# --- BODY SEKM ---
    ax.scatter(
    body_gdf.geometry.x,
    body_gdf.geometry.y,
    color="lightskyblue",
    s=50,
    edgecolors="black",
    zorder=100
    )

    # popisky bodů
    for _, row in body_gdf.iterrows():
        ax.annotate(
            row["Název lokality"][:5],
            (row.geometry.x, row.geometry.y),
            xytext=(3, 3),
            textcoords="offset points",
            fontsize=8,
            zorder=101
        )

    # --- LEGEND ---
    legend_patches = [
        Patch(
            facecolor=my_colors[i],
            edgecolor="black",
            label=f"{bins[i]:.1f}–{bins[i+1]:.1f}"
        )
        for i in range(num_bins)
    ]

    ax.legend(
        handles=legend_patches,
        title="mld CZK",
        loc="upper right",
        frameon=False
    )

    # text 1
    ax.text(**text_1_dict)

    # text 2
    ax.text(**text_2_dict)

    # text 3
    ax.text(**text_3_dict)

    plt.title(f"Částečná distribuce ekologické zátěže po krajích ČR, 2026.\n" \
    f"Vypočtená dle {stats_quantity}ů cen* známých likvidací zátěže z předešlých let.")

    # save
    if save: # means "if save = True"
        safe_stats = stats_quantity.replace("á", "a").replace("ů", "u") # save file name without diacritics
        path = fr"C:/Users/matej.piro/Desktop/cartogram_{safe_stats}.png"
        plt.savefig(path, dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
# map
kraj_price_map = dict(zip(
    Kraje_price_pred_df["Kraj"],
    Kraje_price_pred_df["pred_median_price(mld_CZK)"]
))

my_geo_kraje_df["cena (mld CZK)"] = my_geo_kraje_df["kraj"].map(kraj_price_map)

In [ ]:
create_kraje_cartogram_plus_points(my_geo_kraje_df, my_colors_list, "medián", text_1_dict, text_2_dict, text_3_dict, my_number_of_points = 5, save=False)

## -------------------------------

## MŮŽE SE HODIT

In [ ]:
display(
    mySEKM_DF["Kategorie priority"]
    .value_counts()
    .reset_index()
    .sort_values(by="Kategorie priority")
)

## NADÁLE NEPOTŘEBNÉ

#### Create Cartogram - ORP

##### funkce

In [ ]:
# DEF FUNCTION

# stats_quantity je buď "medián" nebo "průměr"

def create_orp_cartogram(geo_kraje_df, color_list, stats_quantity, text_1_dict, text_2_dict, text_3_dict, save=SAVE_CARTOGRAMS):
    # checks
    if stats_quantity not in ["medián","průměr"]:
        raise ValueError("stats_quantity must be 'medián' or 'průměr'")


    num_bins = 5
    my_colors = color_list

    # --- BINS ---
    bins = np.linspace(
        geo_kraje_df["cena (mld CZK)"].min(),
        geo_kraje_df["cena (mld CZK)"].max(),
        num_bins + 1
    )

    # --- COLORMAP + NORM ---
    cmap = ListedColormap(my_colors)
    norm = BoundaryNorm(bins, len(my_colors))

    # --- PLOT ---
    fig, ax = plt.subplots(figsize=(10, 10))

    # 1) data s hodnotou
    geo_kraje_df.dropna(subset=["cena (mld CZK)"]).plot(
        column="cena (mld CZK)",
        cmap=cmap,
        norm=norm,
        edgecolor="black",
        linewidth=0.8,
        ax=ax
    )

    # 2) NaN data zvlášť (jen hranice)
    geo_kraje_df[geo_kraje_df["cena (mld CZK)"].isna()].plot(
        ax=ax,
        color="none",
        edgecolor="black",
        linewidth=0.8
    )

    # --- LABELS (ORP names) ---
    for idx, row in geo_kraje_df.iterrows():
        if row["geometry"] is not None:
            x = row["geometry"].centroid.x
            y = row["geometry"].centroid.y

            ax.text(
                x, y,
                row["nazev"],
                fontsize=7,
                ha="center",
                va="center"
            )

    # --- LEGEND ---
    legend_patches = [
        Patch(
            facecolor=my_colors[i],
            edgecolor="black",
            label=f"{bins[i]:.1f}–{bins[i+1]:.1f}"
        )
        for i in range(num_bins)
    ]

    legend_patches.append(
    Patch(
        facecolor="white",   # nebo "none"
        edgecolor="black",
        label="0 nebo NaN"
        )
    )

    ax.legend(
        handles=legend_patches,
        title="mld CZK",
        loc="upper left",
        frameon=False
    )

    # text 1
    ax.text(**text_1_dict)

    # text 2
    ax.text(**text_2_dict)

    # text 3
    ax.text(**text_3_dict)

    plt.title(f"Částečná distribuce ekologické zátěže v ORP Ústeckého kraje, 2026.\n" \
    f"Vypočtená dle {stats_quantity}ů cen* známých likvidací zátěže z předešlých let.")

    # save
    if save: # means "if save = True"
        safe_stats = stats_quantity.replace("á", "a").replace("ů", "u") # save file name without diacritics
        path = fr"C:/Users/matej.piro/Desktop/cartogram_{safe_stats}.png"
        plt.savefig(path, dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
# text dicts
text_1_dict = {
    "x": 14.25,
    "y": 50.2,
    "s": "Data: CENIA, SEKM \nVyhotovil: Matěj Píro, 2026",
    "fontsize": 10,
    "color": "black"
}

text_2_dict = {
    "x": 13.3,
    "y": 50.85,
    "s": "Započteni \nnásledující \npůvodci \nznečištění:\n\n" \
    "- plynárenství,\n" \
    "- strojírenství,\n" \
    "- chemický průmysl,\n" \
    "- doprava a distribuce,\n" \
    "- hutnictví a slévárenství",
    "fontsize": 9,
    "color": "black"
}

text_3_dict = {
    "x": 13.5,
    "y": 50.1,
    "s": "*Ceny přepočteny na úroveň roku 2025.",
    "fontsize": 9,
    "color": "black"
}

In [ ]:
my_geo_orp_df = my_geo_orp_df[["OBJECTID","kod","nazev","geometry"]]

##### 1) median

In [ ]:
# map
ORP_price_map = dict(zip(
    ORP_price_pred_df["ORP"],
    ORP_price_pred_df["pred_mean_price(mld_CZK)"]
))

my_geo_orp_df["cena (mld CZK)"] = my_geo_orp_df["nazev"].map(ORP_price_map)

In [ ]:
# filter only ustecky kraj
my_geo_orp_df = my_geo_orp_df[my_geo_orp_df["nazev"].isin(ORP_ustecky)]

In [ ]:
create_orp_cartogram(my_geo_orp_df, my_colors_list, "medián", text_1_dict, text_2_dict, text_3_dict, save=SAVE_CARTOGRAMS)

### -------------------------------

### UŽ NEPOUŽÍVANÉ

### Summing exact "Kc" columns
##### - Ostrava cases (delete?)

In [ ]:
# Kc columns not to sum
KC_columns_not_to_sum = ["odhad_nakladu_cela_lokalita_(Kc)", "SUMA_provedene_prace_(Kc)"]

# select columns to sum
kc_cols = [col for col in my_DF.columns if "(Kc)" in col and col not in KC_columns_not_to_sum]

# create new column with row-wise sum
my_DF["SUMA_provedene_prace_(Kc)"] = my_DF[kc_cols].sum(axis=1)

### Groups aggregation, statistics

In [ ]:
column_to_group_by = "typ_puvodce_znecisteni"

grouped_stats = my_DF.groupby(column_to_group_by).agg(
    count=("odhad_nakladu_cela_lokalita_(Kc)_inf_2025", "count"),
    min =("odhad_nakladu_cela_lokalita_(Kc)_inf_2025", "min"),
    max=("odhad_nakladu_cela_lokalita_(Kc)_inf_2025", "max"),
    mean=("odhad_nakladu_cela_lokalita_(Kc)_inf_2025", "mean"),
    median=("odhad_nakladu_cela_lokalita_(Kc)_inf_2025", "median"),
    sum=("odhad_nakladu_cela_lokalita_(Kc)_inf_2025", "sum")
)

grouped_stats = grouped_stats.reset_index()

# sort by count descending
grouped_stats = grouped_stats.sort_values(by="count", ascending=False)

# flatten columns
grouped_stats.columns = [
    "_".join(col).strip() if isinstance(col, tuple) else col
    for col in grouped_stats.columns
]

# format numbers
num_cols = grouped_stats.select_dtypes(include="number").columns
grouped_stats[num_cols] = grouped_stats[num_cols].applymap(lambda x: "{:,.0f}".format(x))

# display
print(f"See statistics for: '{column_to_group_by}' column in CZK.")
print("-" *10)
grouped_stats.head(15)

### New DF only one locality type

In [ ]:
my_typ_puvodce_znecisteni_code = 4

df_code_X = my_DF[my_DF["typ_puvodce_znecisteni_code"] == my_typ_puvodce_znecisteni_code].copy()
df_code_X.head(1)


## 2) WHOLE SEKM

In [ ]:
# LOAD DATA
Whole_SEKM_file = r"C:\Users\matej.piro\Desktop\PROJEKTY\Suchánek_quest\info_05_02\Export_celý_SEKM_bez_vyloučených_05022026_Matej.xlsx"
my_SEKM_DF = pd.read_excel(Whole_SEKM_file)

my_SEKM_DF.head(1)

### Basic stats

In [ ]:
num_cols = my_SEKM_DF.shape[1]
num_rows = my_SEKM_DF.shape[0]
print("Number of columns:", num_cols)
print("Number of rows:", num_rows)

### Locality types

In [ ]:
# map a number to each typ_puvodce_znecisteni type
categories = sorted(
    my_SEKM_DF["typ_puvodce_znecisteni"].dropna().unique()
)

# dictionary
mapping_dict_SEKM = {cat: i + 1 for i, cat in enumerate(categories)}

# new column "typ_puvodce_znecisteni_code" into DF on second index
my_SEKM_DF.insert(
    1,
    "typ_typ_puvodce_znecisteni_code",
    my_SEKM_DF["typ_puvodce_znecisteni"].map(mapping_dict_SEKM)
)

# get counts of each typ_puvodce_znecisteni type
counts = my_SEKM_DF["typ_puvodce_znecisteni"].value_counts()

# print dictionary with counts
for key, value in mapping_dict_SEKM.items():
    count = counts.get(key, 0)
    print(f"{value} - {key} (count: {count})")